In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import seaborn
import niceplots.utils as nicepl

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import roots_legendre, spherical_jn, j1
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
    "Smooth_resolution": True,
    "nonlinearMatpow": False,
}

h = np.sqrt(0.136 / 0.278)
Omegab = 0.0226 / h**2
Omegam = 0.278
ns = 0.972
As = 2.31e-9
k_pivot = 0.002
mnu = 0.01

cosmodict={
    "h":h,
    "Omegab":Omegab,
    "Omegam":Omegam,
    "mnu":mnu,
    "As": As,
    "k_pivot":k_pivot,
    "ns":ns,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
    "nonlinear_bias": "HMF",
}

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
)

In [ ]:
mycosmo = myssl.current_cosmology
myhalo = myssl.current_halomodel

In [ ]:
nu = 1.901 * u.THz
z = 3
nuObs = nu / (1 + z)
FWHMnu = nuObs / 100
dnu = FWHMnu / np.sqrt(8 * np.log(2))

In [ ]:
astrodict_CCATp={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
}

surveyspecs_CCATp = {
    "Tsys_NEFD": 0 * u.uK, #81 * u.mJy * u.s**0.5, #/ np.sqrt(6912)
    "Nfeeds": 6912,
    "nD": 1,
    "beam_FWHM": 48 * u.arcsec,
    "nu": nu,
    "nuObs": nuObs,
    "Delta_nu": 0,
    "dnu": dnu,
    "tobs": 200000 / 42 * u.h,
    "Omega_field": 0,
    # "do_Jysr": True,
}

In [ ]:
V_rough = np.geomspace(1e6, 1e10, 40) * u.Mpc**3
r = np.power(V_rough / np.pi, 1/3)

Omega_field = (np.pi * r**2 / mycosmo.comoving(z)**2 * u.sr).to(u.deg**2)
Deltanu = (r * mycosmo.Hubble(z) * nuObs / (1 + z)).to(u.GHz)

In [ ]:
def list_to_astro(lst):
    ul = lst[0].unit
    return np.array([li.to(ul).value for li in lst]) * ul


In [ ]:
sigmaV_arr = []
V_arr = []
ssc_arr = []
for Di, Oi in zip(Deltanu, Omega_field):
    spec = surveyspecs_CCATp.copy()
    spec["Delta_nu"] = Di
    spec["Omega_field"] = Oi
    
    pobs_CII = myssl.compute(
    myssl.current_cosmology.cosmopars,
    myssl.current_halomodel.haloparams,
    astrodict_CCATp,
    spec,
    pobs_settings={
        "kmin":3e-2 * u.Mpc**-1,
        "kmax":2 * u.Mpc**-1,
        "nk":50,
    },
    output=["Power spectrum"]
    )["Power spectrum"]

    myastro_CII = pobs_CII.astro
    ssc_cov = scov.SuperSampleCovariance(pobs_CII)
    sigmaV_arr.append(ssc_cov.sigma_survey().squeeze())
    V_arr.append(ssc_cov.survey_specs.Vfield().squeeze())
    ssc_arr.append(ssc_cov.compute_SSC().squeeze())

sigmaV_arr = np.array(sigmaV_arr)
V_arr = list_to_astro(V_arr)
ssc_arr = list_to_astro(ssc_arr)


In [ ]:
ngcov = scov.nonGuassianCov(pobs_CII).compute_nG_Cov().squeeze()
gcov = np.diag(scov.Covariance(pobs_CII).gaussian_nonoise_cov().squeeze()[:, 0, 0])

In [ ]:
k = pobs_CII.k
Pk = pobs_CII.Pk_0bs.squeeze()
normII = 1 / Pk**2 

In [ ]:
kappa = V_arr / V_arr[0]

In [ ]:
fw, fh=plt.rcParams['figure.figsize']
fig, axs=plt.subplots(1,1, figsize=(fw, fh))

fig.suptitle("[CII], $z=3$")


def truncate_colormap(cmap, minval=0.2, maxval=1.0, n=256):
    return LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n))
    )

cmap = truncate_colormap(plt.cm.YlGn, minval=0.20, maxval=1.0)
norm = mpl.colors.LogNorm(
    vmin=kappa[0],
    vmax=kappa[-1]
)

axs.loglog(k, np.diag(gcov) * V_arr[-1] * normII, c=Cs[0], label="Gaussian Cov")
axs.loglog(k, np.diag(ngcov) * V_arr[-1] * normII, c=Cs[1], label="non-Gaussian Cov")

for i, ssc in enumerate(ssc_arr[::5]):
    color = cmap(norm(kappa[::5][i]))
    axs.loglog(
        k,
        V_arr[::5][i] * np.diag(ssc) * normII,
        color=color,
    )
axs.loglog([], [], c=cmap(norm(300)), label="Super Sample Covariance")
axs.legend()

axs.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
axs.set_ylabel(r"$V\,\mathrm{Cov}(k,k)\,P_\mathrm{TT}^{-2}(k) [\mathrm{Mpc}^3]$")
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axs)
cbar.set_label("Volume Scaling Factor")


In [ ]:
R = np.power(3 * V_arr / 4 / np.pi, 1/3)
sigmaV_arr_spherical = myhalo.sigmaR_of_z(R, z)**2

In [ ]:
z = np.atleast_1d(z)

In [ ]:
def W2_cube(k, mu, L):
    phi = np.linspace(-np.pi, np.pi, 300)
    xmod = (L * k / 2).to(1).value
    x = xmod[:, None, None] * np.sqrt(np.clip(1 -mu[None, :, None]**2, 0, 1)) * np.cos(phi)
    y = xmod[:, None, None] * np.sqrt(np.clip(1 -mu[None, :, None]**2, 0, 1)) * np.sin(phi)
    z = xmod[:, None] * mu[None, :]

    phi_int = spherical_jn(0, x)**2 * spherical_jn(0, y)**2
    W = spherical_jn(0, z)**2 * np.trapezoid(phi_int, phi, axis=-1) / (2 * np.pi)
    return W


def sigma_survey_intg(L, Nt=2000, Nmu=150, alpha=2):
    #Transform logk integral into compactified t
    t = np.linspace(0.0, 1.0, Nt)

    scale = L
    k = ((1 / t - 1) ** alpha) / scale

    mu, w = roots_legendre(Nmu)

    jacobian = alpha / (t * (1.0 - t))

    # mu integration
    T2 = W2_cube(k, mu, L)
    T2 = T2.reshape((*k.shape, *mu.shape, *z.shape))
    T2_1d =  np.sum(w[None, :, None] * T2, axis=1) / 2

    # obtain logk integral on t grid
    P = np.reshape(
        mycosmo.matpow(k, z, nonlinear=False, tracer="matter"),
        (*k.shape, *z.shape),
    )
    D = (4 * np.pi * (k[:, None] / (2 * np.pi))**3 * P).to(1).value

    # t integration
    sigma2_intgrnd = D * T2_1d * jacobian[:, None]
    sigma2_intgrnd[~np.logical_and(t>0, t<1), :] = 0.0

    return t, sigma2_intgrnd

def sigma_survey(L, Nt=2000, Nmu=150, alpha=2):
    #Transform logk integral into compactified t
    t, sigma2_intgrnd = sigma_survey_intg(L, Nt, Nmu, alpha)
    sigma2 = np.trapezoid(sigma2_intgrnd, t, axis=0)
    return sigma2.squeeze()

In [ ]:
L = np.power(V_arr, 1/3)

sigmaV_arr_cubic = []
for Li in L:
    sigmaV_arr_cubic.append(sigma_survey(Li))
sigmaV_arr_cubic = np.array(sigmaV_arr_cubic)

In [ ]:
sigmaV_arr

In [ ]:
Gpch = 1000 * myhalo.Mpch 

In [ ]:
plt.loglog(V_arr.to(Gpch**3), V_arr.to(Gpch**3) * mycosmo.growth_factor(1e-5 * u.Mpc**-1, z)**-2 * sigmaV_arr, c=Cs[0], label="Cylindrical")
plt.loglog(V_arr.to(Gpch**3), V_arr.to(Gpch**3) * mycosmo.growth_factor(1e-5 * u.Mpc**-1, z)**-2 * sigmaV_arr_cubic, c=Cs[1], label="Cubic")
plt.loglog(V_arr.to(Gpch**3), V_arr.to(Gpch**3) * mycosmo.growth_factor(1e-5 * u.Mpc**-1, z)**-2 * sigmaV_arr_spherical, c=Cs[2], label="Spherical")
plt.legend()
plt.xlabel(r"$V\,[h^{-3}\,\mathrm{Gpc}^3]$")
plt.ylabel(r"$V\,\sigma_V^2\,D^{-2}\,[h^{-3}\,\mathrm{Gpc}^3]$")

In [ ]:
fw, fh=plt.rcParams['figure.figsize']
fig, axs=plt.subplots(1,2, figsize=(2 * fw, 1.2 * fh), width_ratios=[6, 7])

fig.suptitle("[CII], $z=3$")

axs[0].loglog(V_arr, V_arr * sigmaV_arr, c=Cs[0], label="Cylindrical")
axs[0].loglog(V_arr, V_arr * sigmaV_arr_cubic, c=Cs[1], label="Cubic")
axs[0].loglog(V_arr, V_arr * sigmaV_arr_spherical, c=Cs[2], label="Spherical")
axs[0].legend()
axs[0].set_xlabel("Survey volume $V\,[\mathrm{Mpc}^3]$")
axs[0].set_ylabel(r"$V\,\sigma_V^2[\mathrm{Mpc}^3]$")


def truncate_colormap(cmap, minval=0.2, maxval=1.0, n=256):
    return LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n))
    )

cmap = truncate_colormap(plt.cm.YlGn, minval=0.20, maxval=1.0)
norm = mpl.colors.LogNorm(
    vmin=kappa[0],
    vmax=kappa[-1]
)


for i, ssc in enumerate(ssc_arr[::5]):
    color = cmap(norm(kappa[::5][i]))
    axs[1].loglog(
        k,
        V_arr[::5][i] * np.diag(ssc) * normII,
        color=color,
    )

axs[1].loglog(k, np.diag(gcov) * V_arr[-1] * normII, c=Cs[0], label="Gaussian Cov")
axs[1].loglog(k, np.diag(ngcov) * V_arr[-1] * normII, c=Cs[1], label="non-Gaussian Cov")
axs[1].loglog([], [], c=cmap(norm(300)), label="Super Sample Covariance")

axs[1].legend()
axs[1].set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
axs[1].set_ylabel(r"$V\,\mathrm{Cov}(k,k)\,P_\mathrm{TT}^{-2}(k) [\mathrm{Mpc}^3]$")
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axs[1])
cbar.set_label("Volume Scaling Factor")
plt.savefig("output/scaling_new_plus_shapes.png")


In [ ]:
Deltanu

In [ ]:
Omega_field